# V2M L5 IT & L5 ET — ACh / NE experiment planning (MSc handover)

Focused report for a **2-month MSc project** on V2M pyramidal cells:

- **Region:** V2M only (VISpm + VISam + RSPagl rollup)
- **Cell types:** Allen **L5 IT CTX** and **L5 ET CTX** supertypes (11 total)
- **Priority:** **ACh first** (weeks 1–6), **NE second** (weeks 7–8)

Uses existing supertype synthesis evidence (notebook 05). No re-run of 01–04 required.

Outputs: supertype heatmaps, IT vs ET deltas, pharmacology ordering, transmitter scenarios,
`EXPERIMENT_PLAN_V2M_L5_ACh_NE.txt` with 8-week timeline.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import load_config, resolve_output_dir, start_run
from src.experiment_planning import (
    V2M_L5_IT_ET_SPEC,
    find_latest_synthesis_run,
    run_experiment_planning,
)
from src.utils import print_path

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "query_config.yaml"
config = load_config(CONFIG_PATH)
EXPLORATION_ROOT = resolve_output_dir(cfg=config)

CELL_TYPE_LEVEL = "supertype"
config["cell_type_level"] = CELL_TYPE_LEVEL
SPEC = V2M_L5_IT_ET_SPEC

SYNTHESIS_RUN = find_latest_synthesis_run(EXPLORATION_ROOT, CELL_TYPE_LEVEL)
print_path("Synthesis evidence:", SYNTHESIS_RUN)

OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset="experiment_planning_v2m_l5",
    exploration_root=EXPLORATION_ROOT,
    notebook="07_v2m_l5_experiment_planning",
)
print_path("Output:", OUTPUT_DIR)

In [ ]:
result = run_experiment_planning(
    config,
    synthesis_run=SYNTHESIS_RUN,
    output_dir=OUTPUT_DIR,
    cell_type_level=CELL_TYPE_LEVEL,
    spec=SPEC,
)

summary = result["summary"]
order_table = result["order_table"]
scenario_table = result["scenario_table"]

print(f"Supertype × gene rows: {len(summary):,}")
print(f"Supertypes: {summary['cell_type'].nunique()}")
print(f"  L5 IT: {summary.loc[summary.coarse_type=='L5 IT', 'cell_type'].nunique()}")
print(f"  L5 ET: {summary.loc[summary.coarse_type=='L5 ET', 'cell_type'].nunique()}")
print()
for key, path in sorted(result["paths"].items()):
    print_path(key, path)

In [ ]:
# ACh ordering priorities (primary modulator)
ach = order_table.merge(
    summary[['gene', 'family']].drop_duplicates(), on='gene', how='left'
)
ach = ach[ach.family == 'acetylcholine']
display(
    ach[[
        'cell_type', 'coarse_type', 'gene', 'receptor', 'coupling',
        'confidence_tier', 'expression', 'priority_score',
        'agonists', 'antagonists', 'firing',
    ]].head(20)
)

In [ ]:
# NE ordering (secondary — weeks 7–8)
ne = order_table[order_table['gene'].isin(
    summary.loc[summary.family == 'noradrenaline', 'gene'].unique()
)]
display(
    ne[[
        'cell_type', 'gene', 'receptor', 'confidence_tier',
        'expression', 'priority_score', 'agonists', 'antagonists',
    ]].head(15)
)

## Full text plan

Give the student `EXPERIMENT_PLAN_V2M_L5_ACh_NE.txt` — includes compound list, physiology predictions,
combined ACh/NE scenarios, and an 8-week milestone timeline.

In [ ]:
plan_path = OUTPUT_DIR / SPEC.plan_filename
print(plan_path.read_text(encoding="utf-8")[:6000])